In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller, kpss
from scipy import stats
from arch import arch_model
from pathlib import Path

In [ ]:
DATA_DIR = Path("data")
FIGURE_DIR = Path("figures")

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_curve(nome_file): #Curva_fairmarket.txt or Curve_IR.txt
    df = pd.read_csv(DATA_DIR / nome_file, sep=";")
    df.columns = df.columns.str.strip() #remove with spaces from column names
    df["DT_CALEND"] = pd.to_datetime(df["DT_CALEND"], format="%d/%m/%y") #convert data text to real date dd/mm/yy
    df = df[["ID_CURVA", "GG_NODO", "DT_CALEND", "PC_TASSO"]].copy() #keep only the columns we need
    df = df.rename(columns={"ID_CURVA": "ID_curve", "GG_NODO": "maturity_days", "DT_CALEND": "observation_date", "PC_TASSO": "rate"}) #rename columns
    df["rate"] = df["rate"].astype(float) #convert PC_TASSO to float
    return df.sort_values(["ID_curve", "observation_date", "maturity_days"]).reset_index(drop=True) #sort by ID_curve, observation_date, maturity_days and reset index

def load_spx():
    df = pd.read_csv(f"{DATA_DIR}/SPX.txt", sep=";")
    df.columns = df.columns.str.strip()
    df["DT_CALEND"] = pd.to_datetime(df["DT_CALEND"], format="%d/%m/%y")
    out = df[["DT_CALEND", "IM_PREZZO"]].rename(columns={"DT_CALEND": "observation_date", "IM_PREZZO": "spx_price"})
    out["spx_price"] = out["spx_price"].astype(float)
    return out.sort_values("observation_date").reset_index(drop=True)

def load_ttf():
    df = pd.read_csv(f"{DATA_DIR}/TTF.txt", sep=";", skipinitialspace=True)
    df.columns = df.columns.str.strip()
    df["Date"] = pd.to_datetime(df["Date"].astype(str).str.strip(), format="%d/%m/%Y")
    out = df[["Date", "PX_LAST"]].rename(columns={"Date": "observation_date", "PX_LAST": "ttf"})
    out["ttf"] = out["ttf"].astype(str).str.strip().astype(float)
    return out.sort_values("observation_date").reset_index(drop=True)

def load_eurusd():
    df = pd.read_csv(f"{DATA_DIR}/EURUSD.txt", sep=";")
    df.columns = df.columns.str.strip()
    df["DT_CALEND"] = pd.to_datetime(df["DT_CALEND"], format="%d/%m/%y")
    out = df[["DT_CALEND", "PC_CAMBIO_MID"]].rename(columns={"DT_CALEND": "observation_date", "PC_CAMBIO_MID": "eurusd"})
    out["eurusd"] = out["eurusd"].astype(float)
    return out.sort_values("observation_date").reset_index(drop=True)

In [ ]:
gov    = load_curve("Curva_fairmarket.txt")
ir     = load_curve("Curve_IR.txt")
spx    = load_spx()
ttf    = load_ttf()
eurusd = load_eurusd()
print("values check:")
for nome, df in [("gov", gov), ("ir", ir), ("spx", spx), ("ttf", ttf), ("eurusd", eurusd)]:
    n_nan = int(df.isna().sum().sum()) #missing values
    d_min, d_max = df["observation_date"].min().date(), df["observation_date"].max().date()
    print(f"{nome:7s} | righe={len(df):6d} | date {d_min} -> {d_max} | mancanti={n_nan}")
print("\nvalue range (min/max):")
for nome, df, col in [("spx_price", spx, "spx_price"), ("ttf", ttf, "ttf"), ("eurusd", eurusd, "eurusd")]:
    print(f"  {nome:7s} min={df[col].min():.4f}  max={df[col].max():.4f}")
spx_plot = spx.copy()
spx_plot["spx_price"] = spx_plot["spx_price"].ffill() #forward fill missing values in SPX price series (using the last available value)

#SPX
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(spx_plot["observation_date"], spx_plot["spx_price"], color="tab:blue")
ax.set_title("SPX - indice S&P 500 (USD)")
ax.grid(alpha=0.1)
fig.tight_layout()
plt.show()

#TTF
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ttf["observation_date"], ttf["ttf"], color="tab:orange")
ax.set_title("TTF - gas (EUR)")
ax.grid(alpha=0.1)
fig.tight_layout()
plt.show()

#EUR/USD
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(eurusd["observation_date"], eurusd["eurusd"], color="tab:green")
ax.set_title("EUR/USD - cambio (USD per EUR)")
ax.grid(alpha=0.1)
fig.tight_layout()
plt.show()

In [ ]:
def interpolated_rate(curve_df, valuation_date, days, curve_name=None):
    sel = curve_df[curve_df["observation_date"] == valuation_date] #snapshot of the curve on that day
    if curve_name is not None: #for the IR file: pick ESTR/EUR6M/USDSOFR
        sel = sel[sel["ID_curve"] == curve_name]
    sel = sel.sort_values("maturity_days") #nodes in ascending order
    nodes = sel["maturity_days"].values #maturities (x-axis)
    rates = sel["rate"].values #rates at the nodes (y-axis)
    return np.interp(days, nodes, rates) #linearly interpolated rate

In [ ]:
def years_from_days(days): #convert days to years using ACT(actual)/365 convention since the discounted factor formula uses years and the input nodes are in days.
    return np.asarray(days) / 365

def discount_factor(rate, years): #DF = (1+r)^(-t) where r is the interest rate and t is the time in years
    return (1 + rate) ** (-years) #annual compounding

def df_from_curve(curve_df, valuation_date, days, curve_name=None): 
    r = interpolated_rate(curve_df, valuation_date, days, curve_name) #interpolated rate at the given days
    tau = years_from_days(days) #convert days to years
    return discount_factor(r, tau) #compute the discount factor using the interpolated rate and the time in years

In [ ]:
#BTP

BTP_COUPON       = 0.0365 #annual coupon 3.65%
BTP_FREQ         = 2 #2 payments/year
BTP_MATURITY     = pd.Timestamp("2035-08-01") #maturity date
BTP_FIRST_COUPON = pd.Timestamp("2025-02-01") #first coupon payment date
BTP_FIRST_COUPON_AMOUNT = 0.168614 

def btp_cashflows(valuation_date):
    valuation_date = pd.Timestamp(valuation_date)

    dates = []
    d = BTP_FIRST_COUPON

    while d <= BTP_MATURITY:
        dates.append(d)
        d += pd.DateOffset(months=6)

    cf = pd.DataFrame({"coupon_date": dates})

    cf["coupon_amount"] = 100 * BTP_COUPON / BTP_FREQ  # 1.825
    cf.loc[
        cf["coupon_date"] == BTP_FIRST_COUPON,
        "coupon_amount"
    ] = BTP_FIRST_COUPON_AMOUNT

    cf["amount"] = cf["coupon_amount"]

    # principal only at maturity
    cf.loc[
        cf["coupon_date"] == BTP_MATURITY,
        "amount"
    ] += 100

    cf = cf[cf["coupon_date"] > valuation_date].reset_index(drop=True)
    cf["days"] = (cf["coupon_date"] - valuation_date).dt.days

    return cf


def price_btp(gov_curve, valuation_date):
    cf = btp_cashflows(valuation_date) #future cash flows (dates, days, amounts)
    dfs = df_from_curve(gov_curve, valuation_date, cf["days"].values) #discount factor for each maturity
    cf["df"] = dfs
    cf["pv"] = cf["amount"] * cf["df"] #present value of each flow = amount x DF
    price = cf["pv"].sum() #price = sum of present values
    return price, cf

In [ ]:
def value_on_date(price_df, valuation_date, col): #returns the value of the column 'col' on the specified date
    valuation_date = pd.Timestamp(valuation_date)
    row = price_df[price_df["observation_date"] == valuation_date]
    if row.empty:
        raise ValueError(f"No data for {col} on {valuation_date.date()}")
    return float(row[col].iloc[0])

In [ ]:
#EQUITY SPX pricer

SPX_NOTIONAL_USD = 50_000_000 #50m USD notional (from the email)

def price_equity(spx_df, eurusd_df, valuation_date, entry_date, notional_usd=SPX_NOTIONAL_USD): #value in EUR of the equity position on the given date
    spx_now = value_on_date(spx_df, valuation_date, "spx_price") #SPX level (USD)
    spx_entry = value_on_date(spx_df, entry_date, "spx_price") 
    fx = value_on_date(eurusd_df, valuation_date, "eurusd") #FX rate (USD per EUR)
    value_usd = notional_usd * (spx_now / spx_entry)
    return value_usd / fx

In [ ]:
#TTF FUTURE pricer (commodity gas)

TTF_NOTIONAL_EUR = 10_000_000 #EUR
TTF_LOT_SIZE = 720 #MWh

def price_future(ttf_df, valuation_date, entry_date): #Mark-to-market value of the TTF future position on the given date
    price_now = value_on_date(ttf_df, valuation_date, "ttf") #TTF price today
    price_entry = value_on_date(ttf_df, entry_date, "ttf") #TTF price at entry
    n_lots = round(TTF_NOTIONAL_EUR / (TTF_LOT_SIZE * price_entry))
    return n_lots * TTF_LOT_SIZE * price_now

In [ ]:
#FORWARD EUR/USD pricer

FWD_NOTIONAL_USD = 50_000_000 #USD
FWD_MATURITY     = pd.Timestamp("2026-06-22") #maturity date of the forward (one year after the valuation date)

def fair_forward(ir_df, eurusd_df, valuation_date, maturity):
    days = (pd.Timestamp(maturity) - pd.Timestamp(valuation_date)).days #n. of days from valuation to maturity
    spot   = value_on_date(eurusd_df, valuation_date, "eurusd") #exchange rate (USD per EUR) on the valuation date
    df_eur = df_from_curve(ir_df, valuation_date, days, "ESTR") #discount factor for EUR using the ESTR curve
    df_usd = df_from_curve(ir_df, valuation_date, days, "USDSOFR") #discount factor for USD using the USDSOFR curve
    return spot * df_eur / df_usd #F = S * DF_eur / DF_usd

def price_forward(ir_df, eurusd_df, valuation_date, K, maturity=FWD_MATURITY, notional_usd=FWD_NOTIONAL_USD):
    days = (pd.Timestamp(maturity) - pd.Timestamp(valuation_date)).days
    spot   = value_on_date(eurusd_df, valuation_date, "eurusd")
    df_eur = df_from_curve(ir_df, valuation_date, days, "ESTR")
    df_usd = df_from_curve(ir_df, valuation_date, days, "USDSOFR")

    a_eur = notional_usd / K #EUR you receive at maturity (fixed by K)
    a_usd = notional_usd #USD you pay at maturity
    value_eur = a_eur * df_eur - (a_usd * df_usd) / spot 
    return value_eur

In [ ]:
#ASSET SWAP

def forward_rate(ir_df, valuation_date, days_start, days_end, curve_name="EUR6M"): #Forward rate over the period [days_start, days_end] from the EUR6M curve
    df_start = df_from_curve(ir_df, valuation_date, days_start, curve_name) #DF at period start
    df_end   = df_from_curve(ir_df, valuation_date, days_end, curve_name) #DF at period end
    delta = (days_end - days_start) / 365 #year fraction of the period
    fwd = (df_start / df_end - 1) / delta #forward rate formula
    return fwd

In [ ]:
#ASSET SWAP
ASW_NOTIONAL   = 100_000_000 

def asset_swap_legs(ir_df, valuation_date, spread):
    valuation_date = pd.Timestamp(valuation_date)

    cf = btp_cashflows(valuation_date)[
        ["coupon_date", "days", "coupon_amount"]
    ].copy()

    #start of each period = previous coupon date (for the first one, use valuation date)
    cf["days_start"] = cf["days"].shift(1)
    cf.loc[cf.index[0], "days_start"] = 0 #first period starts today
    cf["days_start"] = cf["days_start"].astype(int)
    cf = cf.rename(columns={"days": "days_end"})

    cf["delta"] = (cf["days_end"] - cf["days_start"]) / 365 #year fraction of each period (ACT/365)
    cf["df"] = df_from_curve(ir_df, valuation_date, cf["days_end"].values, "ESTR") #discount factor (ESTR) for each payment date

    #fixed leg contains the bond coupon only, with no principal repayment.
    cf["fixed"] = cf["coupon_amount"]

    #floating leg: forward Euribor plus the calibrated spread.
    fwd = [forward_rate(ir_df, valuation_date, s, e, "EUR6M")
           for s, e in zip(cf["days_start"], cf["days_end"])]
    cf["fwd_rate"] = fwd
    cf["floating"] = (cf["fwd_rate"] + spread) * cf["delta"] * 100 #per 100 of notional.

    return cf

def calibrate_asset_swap_spread(ir_df, gov_curve, valuation_date):
    cf = asset_swap_legs(ir_df, valuation_date, spread=0.0)

    pv_fixed = (cf["fixed"] * cf["df"]).sum()
    pv_float_no_spread = (cf["floating"] * cf["df"]).sum()
    annuity = (cf["delta"] * cf["df"] * 100).sum()

    bond_price, _ = price_btp(gov_curve, valuation_date)

    spread = (
        pv_fixed
        + (100 - bond_price)
        - pv_float_no_spread
    ) / annuity

    return spread


def price_asset_swap(ir_df, valuation_date, spread, notional=ASW_NOTIONAL):
    cf = asset_swap_legs(ir_df, valuation_date, spread)

    pv_fixed = (cf["fixed"] * cf["df"]).sum()
    pv_floating = (cf["floating"] * cf["df"]).sum()

    value_per_100 = pv_floating - pv_fixed
    value_eur = value_per_100 / 100 * notional

    return value_eur, cf

In [ ]:
def price_portfolio_series(gov_curve, ir_df, spx_df, ttf_df, eurusd_df, start_date="2025-06-23", end_date="2026-06-22", entry_date="2025-06-23"):

    spread = calibrate_asset_swap_spread(ir_df, gov_curve, entry_date)
    K = fair_forward(ir_df, eurusd_df, entry_date, FWD_MATURITY)

    dates = (set(spx_df["observation_date"]) & set(ttf_df["observation_date"]) & set(eurusd_df["observation_date"]))
    dates = sorted(d for d in dates if pd.Timestamp(start_date) <= d <= pd.Timestamp(end_date))

    rows = []
    for D in dates:
        btp, _ = price_btp(gov_curve, D)
        row = {
            "date": D,
            "btp": btp / 100 * 100_000_000,
            "asset_swap": price_asset_swap(ir_df, D, spread=spread)[0],
            "ttf_future": price_future(ttf_df, D, entry_date),
            "equity": price_equity(spx_df, eurusd_df, D, entry_date),
            "forward": price_forward(ir_df, eurusd_df, D, K),
        }
        row["total"] = row["btp"] + row["asset_swap"] + row["ttf_future"] + row["equity"] + row["forward"]
        rows.append(row)

    return pd.DataFrame(rows).set_index("date")

In [ ]:
port = price_portfolio_series(gov, ir, spx, ttf, eurusd)

port_long = price_portfolio_series(gov, ir, spx, ttf, eurusd,
                                    start_date="2023-06-21", end_date="2026-06-22",
                                    entry_date="2025-06-23")

In [ ]:
pnl_long = port_long["total"].diff().dropna()

WINDOW = 510
ALPHA  = 0.01

def logreturn(x):
    if (x <= 0).any():
        raise ValueError("Portfolio values must be strictly positive to compute log-returns.")
    return np.log(x / x.shift(1)).dropna()

logret_long = logreturn(port_long["total"])

In [ ]:
def shock_curve(curve_df, bp=1): #apply a parallel shift of bp basis points to all curve nodes.
    shocked = curve_df.copy()
    shocked["rate"] = shocked["rate"] + bp / 10000
    return shocked

gov_shocked = shock_curve(gov, bp=1)
ir_shocked = shock_curve(ir, bp=1)

#keep the inception asset-swap spread fixed under the rate shock.
spread_sens = calibrate_asset_swap_spread(ir, gov, "2025-06-23")
rows_sens = []
for D in port.index:
    btp_base, _ = price_btp(gov, D)
    btp_shock, _ = price_btp(gov_shocked, D)
    sens_btp = (btp_shock - btp_base) / 100 * 100_000_000
    asw_base = price_asset_swap(ir, D, spread=spread_sens)[0]
    asw_shock = price_asset_swap(ir_shocked, D, spread=spread_sens)[0]
    sens_asw = asw_shock - asw_base
    rows_sens.append({"date": D, "sens_btp": sens_btp, "sens_asw": sens_asw, "combined": sens_btp + sens_asw})

sens_check = pd.DataFrame(rows_sens).set_index("date")
print(sens_check.describe())
print("Correlation:", sens_check["sens_btp"].corr(sens_check["sens_asw"]))

In [ ]:
def stationarity_report(series, name):
    adf_stat, adf_p, *_ = adfuller(series.dropna(), autolag="AIC")
    kpss_stat, kpss_p, *_ = kpss(series.dropna(), regression="c", nlags="auto")
    print(f"{name}")
    print(f"  ADF  : stat={adf_stat:.3f}  p={adf_p:.4f}  -> {'stationary' if adf_p<0.05 else 'NOT stationary'}")
    print(f"  KPSS : stat={kpss_stat:.3f}  p={kpss_p:.4f}  -> {'stationary' if kpss_p>0.05 else 'NOT stationary'}")

stationarity_report(logret_long, "Portfolio log-return")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(logret_long.index, logret_long, lw=.8, color="tab:red")
ax.set_title("Portfolio log-return series", fontsize=16, color="black")
ax.set_xlabel("Date", fontsize=14, color="black")
ax.set_ylabel("Log-return", fontsize=14, color="black")
ax.tick_params(axis="both", labelsize=12, colors="black")
ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "portfolio_logreturns.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(2, 1, figsize=(12, 7))
plot_acf(logret_long, lags=60, ax=ax[0], alpha=None)
ax[0].set_title("ACF of portfolio log-returns", fontsize=16, color="black")
ax[0].set_xlabel("Lag", fontsize=14, color="black")
ax[0].set_ylabel("Autocorrelation", fontsize=14, color="black")
ax[0].tick_params(axis="both", labelsize=12, colors="black")
plot_acf(logret_long.abs(), lags=60, ax=ax[1], alpha=None)
ax[1].set_title("ACF of absolute portfolio log-returns", fontsize=16, color="black")
ax[1].set_xlabel("Lag", fontsize=14, color="black")
ax[1].set_ylabel("Autocorrelation", fontsize=14, color="black")
ax[1].tick_params(axis="both", labelsize=12, colors="black")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "acf_combined.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#VaR and backtesting parameters
ETA_TEST    = 0.05 #significance level for backtesting tests (5%)
REFIT_EVERY = 1 #GARCH refit every 1 days
LAMBDA_BRW  = 0.98 #decay factor for weighted historical simulation
EVT_K       = 40 #number of tail observations used by Hill estimator


def weighted_hs_quantile(returns_window, alpha, lam=LAMBDA_BRW):
    x = np.asarray(returns_window, dtype=float)
    n = len(x)
    age = np.arange(n - 1, -1, -1)
    w = (lam ** age) * (1 - lam) / (1 - lam ** n)
    order = np.argsort(x)
    x_sorted = x[order]
    w_sorted = w[order]
    cum_w = np.cumsum(w_sorted)
    idx = np.searchsorted(cum_w, alpha)
    idx = min(idx, n - 1)
    return float(x_sorted[idx])

#exploratory EVT utilities (not part of the fifteen reported specifications).
def hill_estimator(losses, k):
    order = np.sort(losses)[::-1]
    threshold = order[k]
    top_k = order[:k]
    if threshold <= 0 or np.any(top_k <= 0):
        return np.nan, threshold
    alpha_hat = 1.0 / np.mean(np.log(top_k) - np.log(threshold))
    return alpha_hat, threshold

def evt_quantile(sample, alpha_level, k=EVT_K):
    x = np.asarray(sample, dtype=float)
    n = len(x)
    losses = -x
    alpha_hat, u = hill_estimator(losses, k)
    if np.isnan(alpha_hat):
        return np.nan, np.nan
    p = 1 - alpha_level
    var_loss = u * (k / (n * (1 - p))) ** (1.0 / alpha_hat)
    return -var_loss, alpha_hat

def evt_es_from_var(var_quantile, alpha_hat):
    if np.isnan(var_quantile) or alpha_hat <= 1:
        return np.nan
    var_loss = -var_quantile
    es_loss = var_loss * alpha_hat / (alpha_hat - 1)
    return -es_loss

def hill_plot(sample, k_min=10, k_max=None, title=""):
    losses = -np.asarray(sample, dtype=float)
    n = len(losses)
    if k_max is None:
        k_max = n // 3
    ks = list(range(k_min, k_max))
    alphas = [hill_estimator(losses, k)[0] for k in ks]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(ks, alphas, color="tab:blue")
    ax.axvline(EVT_K, color="tab:red", ls="--", lw=1, label=f"k chosen = {EVT_K}")
    ax.set_xlabel("k (n. of tail observations)"); ax.set_ylabel(r"$\hat{\alpha}$")
    ax.set_title(f"Hill plot -- {title}"); ax.legend(); ax.grid(alpha=.3)
    fig.tight_layout(); plt.show()

In [ ]:
#static window
def compute_var_static_frozen(returns, window=WINDOW, alpha=ALPHA, lam=LAMBDA_BRW, k_evt=EVT_K):
    first_window = returns.iloc[:window]
    z = stats.norm.ppf(alpha)

    hs = np.quantile(first_window, alpha)
    param_gauss = first_window.mean() + first_window.std() * z

    nu, loc, scale = stats.t.fit(first_window * 100)
    param_t = (loc + scale * stats.t.ppf(alpha, df=nu)) / 100

    whs = weighted_hs_quantile(first_window, alpha, lam=lam)

    evt_var, evt_alpha_hat = evt_quantile(first_window, alpha, k=k_evt)
    evt_es = evt_es_from_var(evt_var, evt_alpha_hat)

    out = []
    for t in range(window, len(returns)):
        out.append({
            "date": returns.index[t],
            "actual": returns.iloc[t],
            "static_hs_var": hs,
            "static_gauss_var": param_gauss,
            "static_t_var": param_t,
            "static_whs_var": whs,
            "static_evt_var": evt_var,
            "static_evt_es": evt_es,
        })
    df = pd.DataFrame(out).set_index("date")
    df.attrs["evt_alpha_hat"] = evt_alpha_hat
    return df

bt_static_frozen = compute_var_static_frozen(logret_long)
print(bt_static_frozen.head())

In [ ]:
#static with moving window
def compute_var_static(returns, window=WINDOW, alpha=ALPHA, scale_factor=100):
    z = stats.norm.ppf(alpha)
    out = []
    for t in range(window, len(returns)):
        past = returns.iloc[t-window:t]
        hs = np.quantile(past, alpha)
        param_gauss = past.mean() + past.std() * z
        nu, loc, scale = stats.t.fit(past * scale_factor)
        param_t = (loc + scale * stats.t.ppf(alpha, df=nu)) / scale_factor
        out.append({
            "date": returns.index[t], "actual": returns.iloc[t],
            "hs_var": hs, "param_gauss_var": param_gauss, "param_t_var": param_t,
        })
    return pd.DataFrame(out).set_index("date")

def compute_extra_static_methods(returns, window=WINDOW, alpha=ALPHA, lam=LAMBDA_BRW, k_evt=EVT_K):
    out = []
    for t in range(window, len(returns)):
        past = returns.iloc[t-window:t]
        whs = weighted_hs_quantile(past, alpha, lam=lam)
        evt_var, evt_alpha_hat = evt_quantile(past, alpha, k=k_evt)
        evt_es = evt_es_from_var(evt_var, evt_alpha_hat)
        out.append({"date": returns.index[t], "whs_var": whs, "evt_var": evt_var, "evt_es": evt_es})
    return pd.DataFrame(out).set_index("date")

def ewma_volatility(returns, window=WINDOW, lam=0.94):
    sigma2 = pd.Series(index=returns.index, dtype=float)
    sigma2.iloc[window - 1] = returns.iloc[:window].var()
    for t in range(window, len(returns)):
        sigma2.iloc[t] = lam * sigma2.iloc[t - 1] + (1 - lam) * returns.iloc[t - 1] ** 2
    return np.sqrt(sigma2)

def compute_var_ewma(returns, window=WINDOW, lam=0.94, alpha=ALPHA):
    sigma = ewma_volatility(returns, window, lam)
    z = stats.norm.ppf(alpha)
    return pd.DataFrame({"ewma_var": (z * sigma).iloc[window:]})

bt_moving_base  = compute_var_static(logret_long)
bt_moving_extra = compute_extra_static_methods(logret_long)
bt_moving_ewma  = compute_var_ewma(logret_long)

bt_moving = bt_moving_base.join(bt_moving_extra, how="inner").join(bt_moving_ewma, how="inner")
print(bt_moving.head())


In [ ]:
def compute_var_garch(returns, window=WINDOW, alpha=ALPHA, dist="normal", refit_every=REFIT_EVERY):
    out = []
    res = None
    for i, t in enumerate(range(window, len(returns))):
        past = returns.iloc[t-window:t] * 1000
        if res is None or i % refit_every == 0:
            am = arch_model(past, mean="Constant", vol="GARCH", p=1, q=1, dist=dist, rescale=False)
            res = am.fit(disp="off", show_warning=False)
        fc = res.forecast(horizon=1, reindex=False)
        mu_forc = float(fc.mean.values[-1, 0]) / 1000
        sigma_forc = float(np.sqrt(fc.variance.values[-1, 0])) / 1000
        std_resid = res.resid / res.conditional_volatility
        if dist == "normal":
            z_par = stats.norm.ppf(alpha)
        else:
            nu = res.params["nu"]
            z_par = stats.t.ppf(alpha, df=nu) * np.sqrt((nu - 2) / nu)
        param_var = mu_forc + sigma_forc * z_par
        hs_var = mu_forc + sigma_forc * np.quantile(std_resid, alpha)
        out.append({
            "date": returns.index[t], "actual": returns.iloc[t],
            f"garch_{dist}_param_var": param_var, f"garch_{dist}_hs_var": hs_var,
        })
    return pd.DataFrame(out).set_index("date")

def compute_extra_garch_methods(returns, window=WINDOW, alpha=ALPHA, dist="normal",
                                 refit_every=REFIT_EVERY, lam=LAMBDA_BRW, k_evt=EVT_K):
    out = []
    res = None
    for i, t in enumerate(range(window, len(returns))):
        past = returns.iloc[t-window:t] * 1000
        if res is None or i % refit_every == 0:
            am = arch_model(past, mean="Constant", vol="GARCH", p=1, q=1, dist=dist, rescale=False)
            res = am.fit(disp="off", show_warning=False)
        fc = res.forecast(horizon=1, reindex=False)
        mu_forc = float(fc.mean.values[-1, 0]) / 1000
        sigma_forc = float(np.sqrt(fc.variance.values[-1, 0])) / 1000
        std_resid = res.resid / res.conditional_volatility

        whs_q = weighted_hs_quantile(std_resid, alpha, lam=lam)
        whs_var = mu_forc + sigma_forc * whs_q

        evt_q, evt_alpha_hat = evt_quantile(std_resid, alpha, k=k_evt)
        evt_var = mu_forc + sigma_forc * evt_q if not np.isnan(evt_q) else np.nan
        evt_es_std = evt_es_from_var(evt_q, evt_alpha_hat)
        evt_es = mu_forc + sigma_forc * evt_es_std if not np.isnan(evt_es_std) else np.nan

        out.append({
            "date": returns.index[t],
            f"garch_{dist}_whs_var": whs_var, f"garch_{dist}_evt_var": evt_var, f"garch_{dist}_evt_es": evt_es,
        })
    return pd.DataFrame(out).set_index("date")

bt_gauss_base  = compute_var_garch(logret_long, dist="normal")
bt_gauss_extra = compute_extra_garch_methods(logret_long, dist="normal")
bt_t_base      = compute_var_garch(logret_long, dist="t")
bt_t_extra     = compute_extra_garch_methods(logret_long, dist="t")

bt_dynamic = bt_gauss_base.join(bt_gauss_extra, how="inner") \
                          .join(bt_t_base.drop(columns="actual"), how="inner") \
                          .join(bt_t_extra, how="inner")
print(bt_dynamic.head())

In [ ]:
BLACK = "#000000"

COLOR_MAP = {
    "static_hs_var": "#1B2A4A",
    "hs_var": "#1B2A4A",
    "static_gauss_var": "#C9A227",
    "param_gauss_var": "#C9A227",
    "static_t_var": "#2E8B57",
    "param_t_var": "#2E8B57",
    "static_whs_var": "#B03A2E",
    "whs_var": "#B03A2E",
    "ewma_var": "#6D5C9E",
    "garch_normal_param_var": "#1F77B4",
    "garch_normal_hs_var": "#4FA8DE",
    "garch_normal_whs_var": "#8ECBEE",
    "garch_t_param_var": "#B03A2E",
    "garch_t_hs_var": "#E0685A",
    "garch_t_whs_var": "#F0A99E",
}

LABEL_MAP = {
    "static_hs_var": "HS",
    "static_gauss_var": "Gaussian",
    "static_t_var": "Student-t",
    "static_whs_var": "WHS",
    "hs_var": "HS",
    "param_gauss_var": "Gaussian",
    "param_t_var": "Student-t",
    "whs_var": "WHS",
    "ewma_var": "EWMA",
    "garch_normal_param_var": "GARCH-Normal Parametric",
    "garch_normal_hs_var": "GARCH-Normal HS",
    "garch_normal_whs_var": "GARCH-Normal WHS",
    "garch_t_param_var": "GARCH-Student-t Parametric",
    "garch_t_hs_var": "GARCH-Student-t HS",
    "garch_t_whs_var": "GARCH-Student-t WHS",
}

LINESTYLE_MAP = {
    "static_hs_var": "-",
    "static_gauss_var": "--",
    "static_t_var": "-.",
    "static_whs_var": ":",

    "hs_var": "-",
    "param_gauss_var": "--",
    "param_t_var": "-.",
    "whs_var": ":",
    "ewma_var": (0, (6, 2)),

    "garch_normal_param_var": "-",
    "garch_normal_hs_var": "--",
    "garch_normal_whs_var": ":",
    "garch_t_param_var": "-.",
    "garch_t_hs_var": (0, (5, 2, 1, 2)),
    "garch_t_whs_var": (0, (2, 1)),
}

LINEWIDTH_MAP = {
    "static_hs_var": 2.4,
    "static_gauss_var": 2.4,
    "static_t_var": 2.4,
    "static_whs_var": 2.6,

    "hs_var": 2.3,
    "param_gauss_var": 2.3,
    "param_t_var": 2.3,
    "whs_var": 2.5,
    "ewma_var": 2.5,

    "garch_normal_param_var": 2.5,
    "garch_normal_hs_var": 2.3,
    "garch_normal_whs_var": 2.5,
    "garch_t_param_var": 2.5,
    "garch_t_hs_var": 2.3,
    "garch_t_whs_var": 2.5,
}

MARKER_MAP = {
    "static_hs_var": None,
    "static_gauss_var": None,
    "static_t_var": None,
    "static_whs_var": None,

    "hs_var": None,
    "param_gauss_var": None,
    "param_t_var": None,
    "whs_var": None,
    "ewma_var": None,

    "garch_normal_param_var": "o",
    "garch_normal_hs_var": "s",
    "garch_normal_whs_var": "^",
    "garch_t_param_var": "D",
    "garch_t_hs_var": "v",
    "garch_t_whs_var": "P",
}

def plot_three_panels(bt_static, bt_moving, bt_dynamic):

    panels = [
        (
            bt_static,
            ["static_hs_var", "static_gauss_var", "static_t_var", "static_whs_var"],
            "Static regime"
        ),
        (
            bt_moving,
            ["hs_var", "param_gauss_var", "param_t_var", "whs_var", "ewma_var"],
            "Moving-window regime"
        ),
        (
            bt_dynamic,
            [
                "garch_normal_param_var", "garch_normal_hs_var", "garch_normal_whs_var",
                "garch_t_param_var", "garch_t_hs_var", "garch_t_whs_var"
            ],
            "Dynamic GARCH regime"
        ),
    ]

    fig, axes = plt.subplots(
        3, 1,
        figsize=(15, 16.5),
        sharex=True,
        gridspec_kw={"height_ratios": [4, 5, 6]}
    )

    for ax, (bt, cols, title) in zip(axes, panels):

        ax.bar(
            bt.index,
            bt["actual"],
            color="#666666",
            width=1.0,
            alpha=0.65,
            zorder=1
        )

        for c in cols:
            viol = int((bt["actual"] < bt[c]).sum())

            ax.plot(
                bt.index,
                bt[c],
                color=COLOR_MAP[c],
                linestyle=LINESTYLE_MAP[c],
                linewidth=LINEWIDTH_MAP[c],
                marker=MARKER_MAP[c],
                markersize=4.5 if MARKER_MAP[c] is not None else 0,
                markevery=18 if MARKER_MAP[c] is not None else None,
                markerfacecolor="white" if MARKER_MAP[c] is not None else None,
                markeredgewidth=1.0 if MARKER_MAP[c] is not None else None,
                label=f"{LABEL_MAP[c]}: {viol}",
                zorder=3
            )

        ax.set_title(title, fontsize=19, color=BLACK, fontweight="bold", pad=10)
        ax.set_ylabel("Log-return", fontsize=16, color=BLACK)
        ax.tick_params(axis="both", labelsize=14, colors=BLACK, width=1.0, length=5)
        ax.grid(alpha=0.15, zorder=0)

        for spine in ax.spines.values():
            spine.set_color(BLACK)
            spine.set_linewidth(1.0)

        ax.legend(
            loc="center left",
            bbox_to_anchor=(1.01, 0.5),
            fontsize=13.5,
            framealpha=0.95,
            edgecolor=BLACK,
            labelcolor=BLACK
        )

    axes[-1].set_xlabel("Date", fontsize=16, color=BLACK)

    fig.tight_layout()


    fig.savefig(FIGURE_DIR / "three_panels.png", dpi=300, facecolor="white", bbox_inches="tight")

    plt.show()

plot_three_panels(bt_static_frozen, bt_moving, bt_dynamic)

In [ ]:
def backtransform_to_eur(q, total):
    total_prev = total.shift(1).reindex(q.index)
    return total_prev * (np.exp(q) - 1)

In [ ]:
def backtest_unconditional_coverage(bt, col, alpha=ALPHA, eta=ETA_TEST, method="normal"):
    n = len(bt)
    viol = (bt["actual"] < bt[col]).astype(int)
    x = int(viol.sum())
    pi_hat = x / n
    if method == "normal":
        se = np.sqrt(n * alpha * (1 - alpha))
        stat = (x - n * alpha) / se
        p_value = 2 * (1 - stats.norm.cdf(abs(stat)))
    elif method == "kupiec":
        eps = 1e-10
        pi_c = min(max(pi_hat, eps), 1 - eps)
        ll_null = (n - x) * np.log(1 - alpha) + x * np.log(alpha)
        ll_alt = (n - x) * np.log(1 - pi_c) + x * np.log(pi_c)
        stat = -2 * (ll_null - ll_alt)
        p_value = 1 - stats.chi2.cdf(stat, df=1)
    else:
        raise ValueError("method must be 'normal' or 'kupiec'")
    return {
        "n": n, "violations": x, "violations_rate": pi_hat, "expected": alpha,
        "stat": stat, "p_value": p_value,
        "passes_test": bool(p_value > eta),
        "method": method,
    }

def print_backtest_summary(bt, cols, alpha=ALPHA, eta=ETA_TEST):
    n = len(bt)
    print(f"Total days: {n} | expected violations (~{alpha:.0%}): {alpha*n:.1f}\n")
    rows = []
    for c in cols:
        rn = backtest_unconditional_coverage(bt, c, alpha=alpha, eta=eta, method="normal")
        rk = backtest_unconditional_coverage(bt, c, alpha=alpha, eta=eta, method="kupiec")
        rows.append({
            "method": c, "violations": rn["violations"], "violations_rate": f"{rn['violations_rate']:.2%}",
            "z (normal)": round(rn["stat"], 2), "p-value (normal)": round(rn["p_value"], 3),
            "passes (normal)": rn["passes_test"],
            "LR (Kupiec)": round(rk["stat"], 2), "p-value (Kupiec)": round(rk["p_value"], 3),
            "passes (Kupiec)": rk["passes_test"],
        })
    table = pd.DataFrame(rows).set_index("method")
    print(table.to_string())
    return table

cols_static  = ["static_hs_var", "static_gauss_var", "static_t_var", "static_whs_var"]
cols_moving  = ["hs_var", "param_gauss_var", "param_t_var", "whs_var", "ewma_var"]
cols_dynamic = [c for c in bt_dynamic.columns
                 if c != "actual" and not c.endswith("_es") and "evt" not in c]

print("="*70); print("PANEL 1 - Static window"); print("="*70)
tab_static = print_backtest_summary(bt_static_frozen, cols_static)

print("\n" + "="*70); print("PANEL 2 - Moving window"); print("="*70)
tab_moving = print_backtest_summary(bt_moving, cols_moving)

print("\n" + "="*70); print("PANEL 3 - Dynamic GARCH"); print("="*70)
tab_dynamic = print_backtest_summary(bt_dynamic, cols_dynamic)

In [ ]:
def backtransform_bt_to_eur(bt, total):
    return bt.apply(lambda col: backtransform_to_eur(col, total))

def pinball_score_series(var_forecast, actual, alpha=ALPHA):
    var_forecast = np.asarray(var_forecast, dtype=float)
    actual = np.asarray(actual, dtype=float)
    loss = -actual
    r = -var_forecast
    exceed = (loss > r).astype(float)
    s = (alpha - exceed) * r + exceed * loss
    s = np.where(r > 0, s, 0.0)
    return s

def newey_west_var(x, max_lag=None):
    x = np.asarray(x, dtype=float)
    n = len(x)
    x = x - x.mean()
    if max_lag is None:
        max_lag = max(int(np.floor(4 * (n / 100) ** (2 / 9))), 1)
    gamma0 = np.dot(x, x) / n
    var = gamma0
    for lag in range(1, max_lag + 1):
        gamma = np.dot(x[lag:], x[:-lag]) / n
        w = 1 - lag / (max_lag + 1)
        var += 2 * w * gamma
    return max(var, 1e-12)

def diebold_mariano_test(score_internal, score_standard, eta=ETA_TEST):
    d = np.asarray(score_internal, dtype=float) - np.asarray(score_standard, dtype=float)
    n = len(d)
    dbar = d.mean()
    se = np.sqrt(newey_west_var(d) / n)
    T4 = dbar / se if se > 0 else 0.0
    p_internal_migliore = stats.norm.cdf(T4)
    p_internal_peggiore = 1 - stats.norm.cdf(T4)
    if p_internal_peggiore <= eta:
        zona = "RED"
    elif p_internal_migliore <= eta:
        zona = "GREEN"
    else:
        zona = "YELLOW"
    return {"T4": T4, "score_diff_avg": dbar,
            "p_H0_less_rejected": p_internal_peggiore,
            "p_H0_more_rejected": p_internal_migliore, "zone": zona}

def traffic_light_matrix(scores_dict, eta=ETA_TEST):
    methods = list(scores_dict.keys())
    mat = pd.DataFrame(index=methods, columns=methods, dtype=object)
    for standard in methods:
        for internal in methods:
            if standard == internal:
                mat.loc[standard, internal] = "."
                continue
            res = diebold_mariano_test(scores_dict[internal], scores_dict[standard], eta=eta)
            mat.loc[standard, internal] = res["zone"]
    return mat

bt_static_eur  = backtransform_bt_to_eur(bt_static_frozen, port_long["total"])
bt_moving_eur  = backtransform_bt_to_eur(bt_moving, port_long["total"])
bt_dynamic_eur = backtransform_bt_to_eur(bt_dynamic, port_long["total"])

print("="*70); print("TRAFFIC LIGHT - Panel 1 (static window)"); print("="*70)
scores_static = {c: pinball_score_series(bt_static_eur[c], bt_static_eur["actual"], alpha=ALPHA) for c in cols_static}
tlm_static = traffic_light_matrix(scores_static)
print(tlm_static.to_string())

print("\n" + "="*70); print("TRAFFIC LIGHT - Panel 2 (moving window)"); print("="*70)
scores_moving = {c: pinball_score_series(bt_moving_eur[c], bt_moving_eur["actual"], alpha=ALPHA) for c in cols_moving}
tlm_moving = traffic_light_matrix(scores_moving)
print(tlm_moving.to_string())

print("\n" + "="*70); print("TRAFFIC LIGHT - Panel 3 (dynamic GARCH)"); print("="*70)
scores_dynamic = {c: pinball_score_series(bt_dynamic_eur[c], bt_dynamic_eur["actual"], alpha=ALPHA) for c in cols_dynamic}
tlm_dynamic = traffic_light_matrix(scores_dynamic)
print(tlm_dynamic.to_string())

In [ ]:
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
import numpy as np


def plot_traffic_lights(tlm_static, tlm_moving, tlm_dynamic):

    code = {"GREEN": 0, "YELLOW": 1, "RED": 2, ".": 3}
    cmap = ListedColormap(["#2ecc71", "#f1c40f", "#e74c3c", "#ecf0f1"])

    panels = [
        (
            tlm_static,
            "Static regime",
            {
                "static_hs_var": "HS",
                "static_gauss_var": "Gaussian",
                "static_t_var": "Student-$t$",
                "static_whs_var": "WHS"
            }
        ),
        (
            tlm_moving,
            "Moving-window regime",
            {
                "hs_var": "HS",
                "param_gauss_var": "Gaussian",
                "param_t_var": "Student-$t$",
                "whs_var": "WHS",
                "ewma_var": "EWMA"
            }
        ),
        (
            tlm_dynamic,
            "Dynamic GARCH regime",
            {
                "garch_normal_param_var": "Normal\nParametric",
                "garch_normal_hs_var": "Normal\nHS",
                "garch_normal_whs_var": "Normal\nWHS",
                "garch_t_param_var": "Student-$t$\nParametric",
                "garch_t_hs_var": "Student-$t$\nHS",
                "garch_t_whs_var": "Student-$t$\nWHS"
            }
        )
    ]

    fig, axes = plt.subplots(
    3, 1,
    figsize=(13.5, 15.5),
    gridspec_kw={"height_ratios": [4, 5, 6]}
    )

    for ax, (tlm, title, label_map) in zip(axes, panels):

        methods = tlm.index.tolist()

        grid = np.array([
            [code[tlm.loc[r, c]] for c in methods]
            for r in methods
        ])

        ax.imshow(
            grid,
            cmap=cmap,
            vmin=0,
            vmax=3,
            interpolation="nearest",
            aspect="auto"
        )

        labels = [label_map[m] for m in methods]

        ax.set_xticks(range(len(methods)))
        ax.set_yticks(range(len(methods)))
        ax.set_xticklabels(labels, fontsize=13, color="black")
        ax.set_yticklabels(labels, fontsize=13, color="black")

        # Text inside each cell
        for i in range(len(methods)):
            for j in range(len(methods)):
                val = tlm.iloc[i, j]

                if val != ".":
                    ax.text(
                        j, i, val,
                        ha="center",
                        va="center",
                        fontsize=10,
                        fontweight="bold",
                        color="black"
                    )

        ax.set_title(
            title,
            fontsize=19,
            fontweight="bold",
            color="black",
            pad=10
        )

        ax.set_xlabel(
            "Internal model",
            fontsize=15,
            color="black",
            labelpad=10
        )

        ax.set_ylabel(
            "Standard model",
            fontsize=15,
            color="black",
            labelpad=12
        )

        ax.tick_params(axis="both", colors="black", pad=5)

        for spine in ax.spines.values():
            spine.set_color("black")
            spine.set_linewidth(1.1)

    fig.subplots_adjust(
        hspace=0.58,
        left=0.17,
        right=0.98,
        top=0.98,
        bottom=0.05
    )


    fig.savefig(FIGURE_DIR / "traffic_light_matrices.png", dpi=300, facecolor="white", bbox_inches="tight", pad_inches=0.05)

    plt.show()


plot_traffic_lights(
    tlm_static,
    tlm_moving,
    tlm_dynamic
)

In [ ]:
# ============================================================
# CHAPTER 4 -- FINAL OUTPUTS FOR DISSERTATION
# ============================================================

# ------------------------------------------------------------
# 1. Descriptive statistics of the main return series
# ------------------------------------------------------------

r = logret_long.dropna()

return_stats = pd.Series({
    "Observations":      len(r),
    "Mean":              r.mean(),
    "Standard deviation": r.std(),
    "Minimum":           r.min(),
    "1% quantile":       r.quantile(0.01),
    "Median":            r.median(),
    "Maximum":           r.max(),
    "Skewness":          r.skew(),
    "Excess kurtosis":   r.kurt(),
})

print("=" * 70)
print("DESCRIPTIVE STATISTICS -- LOG-RETURN")
print("=" * 70)
print(return_stats.to_string())


# ------------------------------------------------------------
# 2. Average pinball score for all 15 final VaR specifications
# ------------------------------------------------------------

MODEL_LABELS = {
    # Static
    "static_hs_var":       "HS",
    "static_gauss_var":    "Gaussian",
    "static_t_var":        "Student-t",
    "static_whs_var":      "WHS",

    # Moving
    "hs_var":              "HS",
    "param_gauss_var":     "Gaussian",
    "param_t_var":         "Student-t",
    "whs_var":             "WHS",
    "ewma_var":            "EWMA",

    # Dynamic GARCH
    "garch_normal_param_var": "GARCH-Normal Parametric",
    "garch_normal_hs_var":    "GARCH-Normal HS",
    "garch_normal_whs_var":   "GARCH-Normal WHS",
    "garch_t_param_var":      "GARCH-Student-t Parametric",
    "garch_t_hs_var":         "GARCH-Student-t HS",
    "garch_t_whs_var":        "GARCH-Student-t WHS",
}


def average_pinball_table(scores_dict, cols, regime):
    return pd.DataFrame({
        "Regime": regime,
        "Model": [MODEL_LABELS[c] for c in cols],
        "Average pinball score (EUR)": [
            np.mean(scores_dict[c]) for c in cols
        ]
    })


avg_pinball_all = pd.concat([
    average_pinball_table(
        scores_static,
        cols_static,
        "Static"
    ),
    average_pinball_table(
        scores_moving,
        cols_moving,
        "Moving window"
    ),
    average_pinball_table(
        scores_dynamic,
        cols_dynamic,
        "Dynamic GARCH"
    )
], ignore_index=True)


print("\n" + "=" * 70)
print("AVERAGE PINBALL SCORE - ALL 15 FINAL MODELS")
print("=" * 70)

print(
    avg_pinball_all.to_string(
        index=False,
        formatters={
            "Average pinball score (EUR)": lambda x: f"{x:,.2f}"
        }
    )
)

In [ ]:
cols = ["hs_var", "param_gauss_var", "param_t_var", "whs_var", "ewma_var"]

simple_ret_long = port_long["total"].pct_change().dropna()

print("=== STATIONARITY -- ADF AND KPSS ===\n")
stationarity_report(pnl_long,        "Direct P&L")
print()
stationarity_report(simple_ret_long, "Simple return")

def plot_stationarity_diagnostics(series, name, color="tab:blue"):
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(series.index, series, lw=.8, color=color)
    ax.set_title(name); ax.grid(alpha=.3)
    fig.tight_layout(); plt.show()

    fig, ax = plt.subplots(2, 1, figsize=(11, 7))
    plot_acf(series, lags=60, ax=ax[0], alpha=None); ax[0].set_title(f"ACF -- {name}")
    plot_acf(series.abs(), lags=60, ax=ax[1], alpha=None); ax[1].set_title(f"ACF |.| -- {name} (volatility clustering)")
    fig.tight_layout(); plt.show()

plot_stationarity_diagnostics(pnl_long,        "Direct P&L", color="tab:green")
plot_stationarity_diagnostics(simple_ret_long, "Simple return",     color="tab:blue")

def run_all_moving_models(series):
    base  = compute_var_static(series)
    extra = compute_extra_static_methods(series)
    ewma  = compute_var_ewma(series)
    return base.join(extra, how="inner").join(ewma, how="inner")

bt_pnl    = run_all_moving_models(pnl_long)
bt_simple = run_all_moving_models(simple_ret_long)

def viol_count_pnl(bt):
    return {c: int((bt["actual"] < bt[c]).sum()) for c in cols}

def viol_count_simple(bt):
    total_prev = port_long["total"].shift(1).reindex(bt.index)
    actual_eur = total_prev * bt["actual"]
    return {c: int((actual_eur < total_prev * bt[c]).sum()) for c in cols}

def viol_count_log(bt):
    actual_eur = backtransform_to_eur(
        bt["actual"], port_long["total"]
    )
    return {
        c: int((
            actual_eur <
            backtransform_to_eur(bt[c], port_long["total"])
        ).sum())
        for c in cols
    }

comparison_violations = pd.DataFrame({
    "Direct P&L":         viol_count_pnl(bt_pnl),
    "Simple return": viol_count_simple(bt_simple),
    "Log-return":  viol_count_log(bt_moving),
})
print("\n=== VIOLATIONS BY PORTFOLIO TRANSFORMATION ===")
print(comparison_violations)

def backtransform_to_eur_simple(q, total):
    total_prev = total.shift(1).reindex(q.index)
    return total_prev * q

def backtransform_bt_to_eur_simple(bt, total):
    return bt.apply(lambda col: backtransform_to_eur_simple(col, total))

bt_pnl_eur    = bt_pnl.copy()
bt_simple_eur = backtransform_bt_to_eur_simple(bt_simple, port_long["total"])

scores_pnl    = {c: pinball_score_series(bt_pnl_eur[c],    bt_pnl_eur["actual"],    alpha=ALPHA) for c in cols}
scores_simple = {c: pinball_score_series(bt_simple_eur[c], bt_simple_eur["actual"], alpha=ALPHA) for c in cols}
scores_log    = {c: pinball_score_series(bt_moving_eur[c], bt_moving_eur["actual"], alpha=ALPHA) for c in cols}

tlm_pnl    = traffic_light_matrix(scores_pnl)
tlm_simple = traffic_light_matrix(scores_simple)
tlm_log    = traffic_light_matrix(scores_log)

print("\n=== PAIRWISE CLASSIFICATIONS ACROSS TRANSFORMATIONS ===")
print("Direct P&L == Log-return ?", tlm_pnl.equals(tlm_log))
print("Direct P&L == Simple return:", tlm_pnl.equals(tlm_simple))

avg_loss = pd.DataFrame({
    "Direct P&L":         {c: np.mean(scores_pnl[c])    for c in cols},
    "Simple return": {c: np.mean(scores_simple[c]) for c in cols},
    "Log-return":  {c: np.mean(scores_log[c])    for c in cols},
})
print("\nAverage pinball score by model and transformation:")
print(avg_loss)